# 1. Import

필요한 라이브러리를 불러오고 `constant_velocity` 베이스라인을 시작합니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# 2. 데이터 로드

최종 데이터셋 경로를 설정하고, 학습/평가 샘플 및 라벨 파일을 불러옵니다.


In [ ]:
DATA_DIR = Path('./')
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
TRAIN_LABELS_PATH = DATA_DIR / 'train_labels.csv'
SAMPLE_SUBMISSION_PATH = DATA_DIR / 'sample_submission.csv'

train_labels = pd.read_csv(TRAIN_LABELS_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

train_files = sorted(TRAIN_DIR.glob('TRAIN_*.csv'))
test_files = sorted(TEST_DIR.glob('TEST_*.csv'))

print(f'train files: {len(train_files)}')
print(f'test files: {len(test_files)}')
train_labels.head()

# 3. 데이터 전처리 및 보조 함수

각 샘플의 마지막 두 시점을 이용해 최근 40ms 이동 벡터를 2배 외삽하여 80ms 뒤 좌표를 예측합니다.


In [ ]:
R_HIT = 0.01

def constant_velocity_predict(sample_path: Path):
    df = pd.read_csv(sample_path)
    prev_xyz = df.loc[df.index[-2], ['x', 'y', 'z']].to_numpy(dtype=float)
    last_xyz = df.loc[df.index[-1], ['x', 'y', 'z']].to_numpy(dtype=float)
    pred_xyz = last_xyz + 2.0 * (last_xyz - prev_xyz)
    return pred_xyz

def build_prediction_df(sample_files):
    rows = []
    for sample_path in sample_files:
        pred_xyz = constant_velocity_predict(sample_path)
        rows.append({
            "id": sample_path.stem,
            "x": pred_xyz[0],
            "y": pred_xyz[1],
            "z": pred_xyz[2],
        })
    return pd.DataFrame(rows)


# 4. 학습 데이터 기준 베이스라인 성능 확인

학습 데이터에서 `constant_velocity` 예측을 만든 뒤 `R_hit = 0.01m` 기준 hit rate를 확인합니다.


In [ ]:
train_pred = build_prediction_df(train_files)
train_eval = train_labels.merge(train_pred, on='id', suffixes=('_true', '_pred'))

true_xyz = train_eval[['x_true', 'y_true', 'z_true']].to_numpy()
pred_xyz = train_eval[['x_pred', 'y_pred', 'z_pred']].to_numpy()
distance = np.linalg.norm(true_xyz - pred_xyz, axis=1)
hit_rate = np.mean(distance <= R_HIT)

print(f'Constant Velocity Train Hit Rate @ {R_HIT:.3f}m: {hit_rate:.4f}')
print(f'Mean Distance Error: {distance.mean():.6f} m')


# 5. 테스트 데이터 예측 생성

평가용 샘플 전체에 대해 `constant_velocity` 방식 예측을 생성합니다.


In [ ]:
test_pred = build_prediction_df(test_files)
test_pred.head()

# 6. submission.csv 생성

`sample_submission.csv` 형식에 맞춰 최종 제출 파일을 생성합니다.


In [ ]:
submission = sample_submission[['id']].merge(test_pred, on='id', how='left')
submission.to_csv('./submission.csv', index=False)
print('saved: submission.csv')
submission.head()